## Spatial Data Science (GIS6307/GEO4930)


<br>
Instructor: Yi Qiang (qiangy@usf.edu)<br>
Teaching Assistant: Jinwen Xu (jinwenxu@usf.edu)

---

# Workshop on Spatial Analysis of Twitter

This workshop will help you to get started with the acquisition, processing, and analysis of Twitter data using data science techniques. Specifically, you will learn:

- Streaming real-time tweets using Twitter Developer APIs.
- Processing the raw tweets into an analyzable form.
- Basic mapping, spatial analysis and natural language processing for Twitter data.

### Prerequisite
- Install Anaconda in your computer.
- Activation of Twitter Developer Account and approved **Elevated Access** before the workshop.
- Basic programming skills are recommended, but not required.




## 1. Install Python Libraries

We will need two new libraries [tweepy](https://www.tweepy.org/) and [folium](https://plotly.com/python/) for this lab. Please do the following steps to install these two libarries.

1. Please open Anaconda Prompt, and use the command `conda activate geo` to activate the "geo" environment that you created in the previous lab. 

2. Install tweepy using the following command:

    `conda install -c conda-forge tweepy`
    
3. Install contextily using the following command:

    `conda install -c conda-forge folium`
    
4. Run the following code to import the installed libraries and other needed libraries. If the code runs through, the libraries are installed successfully.

In [37]:
# Run the following lines if there is an error loading basemap
# import os
# os.environ['PROJ_LIB'] = '~your anaconda 3 path/Anaconda3/Library/share/'

import tweepy
import folium
import pandas as pd

#from mpl_toolkits.basemap import Basemap
import matplotlib.pyplot as plt

## 2. Set-Up the Connection to Twitter APIs

Go to Twitter Developer Portal (https://developer.twitter.com/en/apps). Click the App you created in account activation.

If you haven't created an App when you created the account, you can create one in the project.

Click `Sign up`.

![](../image/twitter/signup3.jpg)

Turn on `OAuth 1.0a` and keep `OAuth 2.0` off. 

![](../image/twitter/OAuth.jpg)

Select "Read and write and Direct message". You can use "http://127.0.0.1:8080" as Callback URL. Add any website as the website URL (e.g. your personal website or https://google.com).

![](../image/twitter/setting.jpg)

Copy the API keys you have saved when you activated your Developer account, and paste them to replace "......" below. If you can't find them, you can **regenerate** the keys and tokens in your Developer Portal. 

![](../image/twitter/keys2.jpg)

Generate your Access Token and Secret and paste them below.

In [38]:
# paste your API keys and tokens to replace ......
# API_key = 'aOLqeAcCKvMRhKLZBIKADW2rW'
# API_key_secret = 'b1lOWTIy3Vfb5C2OS4S9wXP3BYzugrPs50i4oIqWmn1tcdORaT'
# access_token = '1508096832806731780-wIo5pKdrys6o1qoX7LETloH1tRirkS'
# access_token_secret = 'U4g0XQiGxGXnwN2Y5T2ePqKdxhS91jcHy2Zc5jqBEJ9V8'

API_key = 'gmDPt4mHl24zKIWeabwu1ebDs'
API_key_secret = '2Wx9lIi37wAH8oCHyS6Lw5dzboif7B8xxjMP3lnaGsKCIbtoEC'
access_token = '1343615702809374721-nBEV6K4qlSvkUFWpxhjn2Q3Gf5WpaL'
access_token_secret = 'z2dTOARM5FpdIMqOP9IzRwM9bzCvFNh9UhgvOD8WkquTc'

Set up for Twitter authentication.

In [39]:
auth = tweepy.OAuthHandler(API_key, API_key_secret)
auth.set_access_token(access_token, access_token_secret)

Set up tweepy API and set rate limit to be true.

In [40]:
api = tweepy.API(auth, wait_on_rate_limit=True)

---

## 3. Simple Operations with Twitter APIs

Now, your working environment is set up for Twitter analysis. Let's first try a few simple operations to acquire Twitter data in a programmatic way.

The full functionalities of Twitter API and Tweepy can be found in:

- [Twitter APIs](https://developer.twitter.com/en/docs.html)
- [Tweepy documentation](http://docs.tweepy.org/en/v4.8.0/)

### 3.1 Posting/Deleting a Tweet

First, let's post a message in your Twitter account.

**Note**: if you don't want to disturb your followers with a meanless tweet, don't run the following block of code.

In [41]:
# Post a tweet from Python
test_tweet = api.update_status("DRILL: I'm creating a robot to tweet!")

Check your Twitter account, and you'll see the above message is posted.

![](../image/twitter/tweet.jpg)

Tweets are encoded in a JSON (JavaScript Object Notation) format. You can run the following code to check the content of the tweet you just posted.

In [42]:
test_tweet._json

{'created_at': 'Thu Mar 31 15:30:39 +0000 2022',
 'id': 1509553769419542530,
 'id_str': '1509553769419542530',
 'text': "DRILL: I'm creating a robot to tweet!",
 'truncated': False,
 'entities': {'hashtags': [], 'symbols': [], 'user_mentions': [], 'urls': []},
 'source': '<a href="https://google.com" rel="nofollow">GIS6307</a>',
 'in_reply_to_status_id': None,
 'in_reply_to_status_id_str': None,
 'in_reply_to_user_id': None,
 'in_reply_to_user_id_str': None,
 'in_reply_to_screen_name': None,
 'user': {'id': 1343615702809374721,
  'id_str': '1343615702809374721',
  'name': 'alex',
  'screen_name': 'alex32348132',
  'location': '',
  'description': 'me',
  'url': None,
  'entities': {'description': {'urls': []}},
  'protected': False,
  'followers_count': 0,
  'friends_count': 9,
  'listed_count': 2,
  'created_at': 'Mon Dec 28 17:52:11 +0000 2020',
  'favourites_count': 1,
  'utc_offset': None,
  'time_zone': None,
  'geo_enabled': False,
  'verified': False,
  'statuses_count': 1,
  'l

`_json` returns a dictionary object. So you could access specific attributes using keys in the dictionary. The code below gets the posting time of the tweets.

In [43]:
test_tweet._json['created_at']

'Thu Mar 31 15:30:39 +0000 2022'

Alternative, you can also use the built-in attribute of the tweepy status object to access the attribute. All attributes of a tweet can be found [here](https://developer.twitter.com/en/docs/twitter-api/v1/data-dictionary/object-model/tweet).

In [44]:
test_tweet.created_at

datetime.datetime(2022, 3, 31, 15, 30, 39, tzinfo=datetime.timezone.utc)

You can run the following code to delete the drill tweet you just posted.

In [45]:
api.destroy_status(test_tweet.id_str)

Status(_api=<tweepy.api.API object at 0x0000022E113CD880>, _json={'created_at': 'Thu Mar 31 15:30:39 +0000 2022', 'id': 1509553769419542530, 'id_str': '1509553769419542530', 'text': "DRILL: I'm creating a robot to tweet!", 'truncated': False, 'entities': {'hashtags': [], 'symbols': [], 'user_mentions': [], 'urls': []}, 'source': '<a href="https://google.com" rel="nofollow">GIS6307</a>', 'in_reply_to_status_id': None, 'in_reply_to_status_id_str': None, 'in_reply_to_user_id': None, 'in_reply_to_user_id_str': None, 'in_reply_to_screen_name': None, 'user': {'id': 1343615702809374721, 'id_str': '1343615702809374721', 'name': 'alex', 'screen_name': 'alex32348132', 'location': '', 'description': 'me', 'url': None, 'entities': {'description': {'urls': []}}, 'protected': False, 'followers_count': 0, 'friends_count': 9, 'listed_count': 2, 'created_at': 'Mon Dec 28 17:52:11 +0000 2020', 'favourites_count': 1, 'utc_offset': None, 'time_zone': None, 'geo_enabled': False, 'verified': False, 'statuse

### 3.2 Getting Trending Tweets

Get the list of cities where trends are available

In [46]:
city_ls = api.available_trends()

Convert the list (in JSON format) into a dataframe (i.e. a table). Print the number of cities where trends are available.

In [47]:
df_city = pd.DataFrame(city_ls)

print (str(len(df_city)) + " cities have trends.")

467 cities have trends.


Preview 10 cities where trends are available

In [48]:
df_city.head(10)

,name,placeType,url,parentid,country,woeid,countryCode
0,Worldwide,"{'code': 19, 'name': 'Supername'}",http://where.yahooapis.com/v1/place/1,0,,1,None
1,Winnipeg,"{'code': 7, 'name': 'Town'}",http://where.yahooapis.com/v1/place/2972,23424775,Canada,2972,CA
2,Ottawa,"{'code': 7, 'name': 'Town'}",http://where.yahooapis.com/v1/place/3369,23424775,Canada,3369,CA
3,Quebec,"{'code': 7, 'name': 'Town'}",http://where.yahooapis.com/v1/place/3444,23424775,Canada,3444,CA
4,Montreal,"{'code': 7, 'name': 'Town'}",http://where.yahooapis.com/v1/place/3534,23424775,Canada,3534,CA
5,Toronto,"{'code': 7, 'name': 'Town'}",http://where.yahooapis.com/v1/place/4118,23424775,Canada,4118,CA
6,Edmonton,"{'code': 7, 'name': 'Town'}",http://where.yahooapis.com/v1/place/8676,23424775,Canada,8676,CA
7,Calgary,"{'code': 7, 'name': 'Town'}",http://where.yahooapis.com/v1/place/8775,23424775,Canada,8775,CA
8,Vancouver,"{'code': 7, 'name': 'Town'}",http://where.yahooapis.com/v1/place/9807,23424775,Canada,9807,CA
9,Birmingham,"{'code': 7, 'name': 'Town'}",http://where.yahooapis.com/v1/place/12723,23424975,United Kingdom,12723,GB


Get the record of Tampa. The `woeid` is a unique ID for each place.

In [49]:
df_city[df_city['name']=='Tampa']

,name,placeType,url,parentid,country,woeid,countryCode
394,Tampa,"{'code': 7, 'name': 'Town'}",http://where.yahooapis.com/v1/place/2503863,23424977,United States,2503863,US


Store the `woeid` of Tampa in tampa_id.

In [50]:
tampa_id = df_city[df_city['name']=='Tampa']['woeid']

Return the trends in Tampa.

Note: you need to convert the city_id from a pandas series object into an integer.

In [51]:
# make Tampa as an example
trends_tampa = api.get_place_trends(int(tampa_id))

Print the trends in JSON format

In [52]:
# print the top 20 trends in Tampa
trends_tampa[0:20]

[{'trends': [{'name': 'Arians',
    'url': 'http://twitter.com/search?q=Arians',
    'promoted_content': None,
    'query': 'Arians',
    'tweet_volume': 45626},
   {'name': '#Bucs',
    'url': 'http://twitter.com/search?q=%23Bucs',
    'promoted_content': None,
    'query': '%23Bucs',
    'tweet_volume': None},
   {'name': '#Cuba',
    'url': 'http://twitter.com/search?q=%23Cuba',
    'promoted_content': None,
    'query': '%23Cuba',
    'tweet_volume': 99931},
   {'name': '#TransDayOfVisibility',
    'url': 'http://twitter.com/search?q=%23TransDayOfVisibility',
    'promoted_content': None,
    'query': '%23TransDayOfVisibility',
    'tweet_volume': 95165},
   {'name': '#TDOV',
    'url': 'http://twitter.com/search?q=%23TDOV',
    'promoted_content': None,
    'query': '%23TDOV',
    'tweet_volume': 31703},
   {'name': 'Wordle 285 X',
    'url': 'http://twitter.com/search?q=%22Wordle+285+X%22',
    'promoted_content': None,
    'query': '%22Wordle+285+X%22',
    'tweet_volume': None}

Organize the Tampa trends in a dataframe.

In [53]:
trend_ls = [[trend['name'], trend['url'], trend['tweet_volume']] for trend in trends_tampa[0]['trends']]

df_trends = pd.DataFrame(trend_ls,columns=['name','url','tweet_volume'])

Sort the trends by tweet volumn in a descending order and print the top 10 trends with the most tweeting volumne.

In [54]:
# Sort the trends by tweet volumn in a descending order
df_trends.sort_values("tweet_volume", inplace = True, ascending = False)

# Print the top 10 trends ranked by tweet volumne
df_trends.head(10)

,name,url,tweet_volume
12,April Fools,http://twitter.com/search?q=%22April+Fools%22,108681.0
26,mnet,http://twitter.com/search?q=mnet,102901.0
2,#Cuba,http://twitter.com/search?q=%23Cuba,99931.0
3,#TransDayOfVisibility,http://twitter.com/search?q=%23TransDayOfVisib...,95165.0
41,Ivanka,http://twitter.com/search?q=Ivanka,90680.0
13,Morbius,http://twitter.com/search?q=Morbius,71227.0
25,GFRIEND,http://twitter.com/search?q=GFRIEND,63071.0
18,The Need To Know,http://twitter.com/search?q=%22The+Need+To+Kno...,61726.0
42,VIVA LA VIVIZ,http://twitter.com/search?q=%22VIVA+LA+VIVIZ%22,48788.0
0,Arians,http://twitter.com/search?q=Arians,45626.0


The table shows the popular topics people are tweeting about in Tampa.

---

## 4. Acquiring Tweets using the Search API

### 4.1 Search Tweets using Keywords

In this step, you will use Python program to search tweets that contain a specific keyword. "Will Smith" was quite a hot topic when I created the tutorial. Next, I will search for Tweets that contain "Will Smith".

In [55]:
tweets = api.search_tweets("Will Smith",count=100)

print("Total retweet retrieved: "+ str(len(tweets)))

Total retweet retrieved: 100


Store the user name, user location, posting time, and tweet text in a Pandas dataframe.

In [56]:
tweets_pd = pd.DataFrame([[tweet.user.name, tweet.user.location,tweet.created_at, tweet.text] for tweet in tweets], 
                         columns = ['user_name','user_loc','creation_time','text'])

tweets_pd

,user_name,user_loc,creation_time,text
0,NanB,,2022-03-31 15:30:42+00:00,@TheView Ban will smith from the academy .. pr...
1,gaia (pro-slap),t(he)y 21,2022-03-31 15:30:42+00:00,RT @jiggyjayy2: Idk they gave Mel Gibson an Os...
2,#ConfirmJudgeJackson Pass #VotingRightAct,CA,2022-03-31 15:30:42+00:00,RT @kenolin1: It shouldn’t surprise anyone tha...
3,scotoma,USA,2022-03-31 15:30:42+00:00,@GeorgeEdelman So this package would make Will...
4,Beat A Dead Source,"Cleveland, OH",2022-03-31 15:30:42+00:00,https://t.co/C5t6FZl713\nOk ok ok hold on. How...
...,...,...,...,...
95,James Renshaw,,2022-03-31 15:30:26+00:00,RT @Durham_isComing: Jussie smollett should ha...
96,Maureen L. #MAGA 🇺🇸,"Florida, USA",2022-03-31 15:30:26+00:00,Chris Rock 'still processing' Will Smith Oscar...
97,FOX 7 Austin,"Austin, TX",2022-03-31 15:30:26+00:00,The Academy said Will Smith was asked to leave...
98,Vell Gotti 🕴🏾,on a cloud above Cincinnati,2022-03-31 15:30:25+00:00,RT @CultureCrave: Daniel Radcliffe when asked ...


The `search_tweets` funciton can retrieve max 100 tweets at one time. If you want to get more tweets, you can use the `cursor`.

The following code can retrieve more tweets containing a keyword. Here we still retrieve 100 tweets to save your search quota for the following steps. You could try to increase the `num` variable when testing at home.

> Note: You can only retrieve a limited number of tweets per 15 minutes. If the retrieved tweets exceed the limit, the program will pause for some time. If you can't wait, you can doublepress `i` on your keyboard to interrupt the process.

In [57]:
# Number of tweets to be retrieved.
num = 100

# define the search keyword
keyword = "Will Smith"

# use cursor to send your request with parameters
tweets = tweepy.Cursor(api.search_tweets,
                   q = keyword,
                   count = num,
                   lang="en").items(num)

# restore the results as a list
tweet_ls = [[tweet.user.name, tweet.user.location,tweet.created_at, tweet.text] for tweet in tweets]

# Store the retrieved tweets in a dataframe
tweets_pd_full = pd.DataFrame(tweet_ls, 
                         columns = ['user_name','user_loc','creation_time','text'])

# Print the dataframe
tweets_pd_full

,user_name,user_loc,creation_time,text
0,Daiguren Hyōrinmaru,,2022-03-31 15:30:43+00:00,RT @jiggyjayy2: Idk they gave Mel Gibson an Os...
1,Bourgeoisie (Engr.),"South South, Nigeria",2022-03-31 15:30:43+00:00,RT @piersmorgan: Great to see Hollywood so inc...
2,theworldisendingandwearetripping,,2022-03-31 15:30:43+00:00,RT @ThePopPunkDad: That time in 1991 when Will...
3,Lisa Goranson,Chicago suburbs.,2022-03-31 15:30:43+00:00,"@TheView I'm not getting Whoopies point of ""vi..."
4,🕵️‍♂️🥑no limb angel🦠🧙‍♂️,,2022-03-31 15:30:43+00:00,@cwmmies slap me Like will smith… jk sry
...,...,...,...,...
95,Mamba Mentality,New Jersey,2022-03-31 15:30:24+00:00,RT @Phil_Lewis_: The Academy lied about asking...
96,J. R. Tomlin,Oregon USA by way of Scotland,2022-03-31 15:30:24+00:00,RT @Camz99: Rishi Sunak compares himself to Wi...
97,Plumduffstuff,,2022-03-31 15:30:24+00:00,@TheAcademy Will Smith demonstrated complete d...
98,✊🏽💞,success dr. off ambition st.,2022-03-31 15:30:24+00:00,"RT @JwinHitz: ""Will Smith set black people bac..."


### 4.2 Removing Retweets
The retrieved tweets include original tweets and retweets. The textual content of retweets are almost identical. You can set up a filter to eliminate the retweets and keep only the original tweets.

In [58]:
new_keyword = "Will Smith" + " -filter:retweets"
new_keyword

'Will Smith -filter:retweets'

Now, you can see only original tweets are retrieved.

Print to see the list.

In [59]:
# Number of tweets to be retrieved.
num = 100

# use cursor to send your request with parameters
tweets = tweepy.Cursor(api.search_tweets,
                   q = new_keyword,
                   count = num,
                   lang="en").items(num)

# restore the results as a list
tweet_ls = [[tweet.user.name, tweet.user.location,tweet.created_at, tweet.text] for tweet in tweets]

# Store the retrieved tweets in a dataframe
tweets_pd_full = pd.DataFrame(tweet_ls, 
                         columns = ['user_name','user_loc','creation_time','text'])

# Print the dataframe
tweets_pd_full

,user_name,user_loc,creation_time,text
0,VAN NO GOGH,3500,2022-03-31 15:30:44+00:00,Just learned Will Smith name is Willard.. idk ...
1,Gaviria,Arsenal,2022-03-31 15:30:43+00:00,@AllLove4Philly @Phil_Lewis_ Who's fixed this?...
2,Lisa Goranson,Chicago suburbs.,2022-03-31 15:30:43+00:00,"@TheView I'm not getting Whoopies point of ""vi..."
3,🕵️‍♂️🥑no limb angel🦠🧙‍♂️,,2022-03-31 15:30:43+00:00,@cwmmies slap me Like will smith… jk sry
4,NanB,,2022-03-31 15:30:42+00:00,@TheView Ban will smith from the academy .. pr...
...,...,...,...,...
95,Maureen L. #MAGA 🇺🇸,"Florida, USA",2022-03-31 15:29:45+00:00,"Will Smith refused to leave Oscars, broke cond..."
96,Spurs Legacy 🦦,,2022-03-31 15:29:44+00:00,"Finally a good Will Smith take, thank you Radc..."
97,Juwan,"Fredericksburg, VA",2022-03-31 15:29:43+00:00,Will Smith is in the audience \nComedian: Oh m...
98,Rahul Attri,Jammu/Noida,2022-03-31 15:29:43+00:00,@ArvindKejriwal Will Smith is being missed .🤣


### 4.3 Search Tweets using locations

Query for a popular trend keyword in Tampa (200 miles range).

First, let's check what are the top 10 trending topics in the selected city (Tampa).

The following code may take a few minutes to run to collect the tweets, depending on the number of tweets.

In [60]:
# Number of tweets to be retrieved.
num = 200

# define the search keyword
keyword = "Will Smith"

# use cursor to send your request with parameters
tweets = tweepy.Cursor(api.search_tweets,
                   q=keyword,
                   geocode = "28.0619,-82.4146,20mi",
                   count = num,
                   lang="en").items(num)

# restore the results as a list
tweet_ls = [[tweet.user.name, tweet.user.location, tweet.place, tweet.created_at, tweet.text] for tweet in tweets]

# Store the retrieved tweets in a dataframe
tweets_pd_geo = pd.DataFrame(tweet_ls, 
                         columns = ['user_name','user_loc','geotag','creation_time','text'])

# Print the dataframe
tweets_pd_geo

,user_name,user_loc,geotag,creation_time,text
0,BREEZY,"Tampa, FL",None,2022-03-31 15:12:35+00:00,*mutes Will Smith &amp; Chris Rock*
1,Ashley Stotts,"Tampa, FL",None,2022-03-31 14:56:20+00:00,@atownsquare He doesn't need to. The backlash ...
2,Kathryn Cardozo,"Tampa, FL",None,2022-03-31 14:46:57+00:00,"@kenolin1 Even without delving deep, I’m disap..."
3,Fluidity Balance & Scar Tissue Release,"Tampa, FL",None,2022-03-31 14:21:40+00:00,@KelleyLCarter Drama. Mr. Rock is not pressing...
4,P.Salvati Norris,Clearwater. Florida,None,2022-03-31 14:19:12+00:00,Process it and sue his ass..Will Smith assault...
...,...,...,...,...,...
195,kp 🏳️‍🌈,"Clearwater, FL",None,2022-03-30 05:16:14+00:00,@melissamalcolm7 @Variety So being humiliated/...
196,Luke,"Tampa, FL",None,2022-03-30 04:43:38+00:00,@kabongodan99 @zenromanov She literally just r...
197,Keetos 🦧,"Tampa, FL",None,2022-03-30 04:24:31+00:00,🚨 Geico Lizard shares his thoughts on Will Smi...
198,10 Tampa Bay,"Tampa, Florida",None,2022-03-30 03:42:55+00:00,Will Smith’s slap seen ’round the world at the...


Check how many tweets have geotags.

In [61]:
all = len(tweets_pd_geo[tweets_pd_geo['geotag'].notna()]) # all retrieved tweets
geo = len(tweets_pd_geo) # tweets that actually have geotags.

print("%s out of the %s retrieved tweets actually have geotags" % (all, geo))

7 out of the 200 retrieved tweets actually have geotags


#### Copy tweets with geotags to a new dataframe called "geotags"

In [62]:
geo_tweets = tweets_pd_geo.loc[tweets_pd_geo['geotag'].notna()].copy()

#### get their place and view where first 5 tweets are from

In [63]:
# extract place name from the geotags
geo_tweets['place_name'] = geo_tweets.geotag.apply(lambda s:s.name)

# Print updated geo_tweets
geo_tweets

,user_name,user_loc,geotag,creation_time,text,place_name
12,I Am The One Who Knocks,South of Heaven,Place(_api=<tweepy.api.API object at 0x0000022...,2022-03-31 13:37:48+00:00,@paulduanefilm Robin Williams.\nWill Smith.\nT...,Tampa
23,444,"Florida, USA",Place(_api=<tweepy.api.API object at 0x0000022...,2022-03-31 12:15:56+00:00,Defend Will Smith although he was wrong he’s a...,Tampa
50,#UVA 🍺🏈⚾️🏀🏒🐴🌴🌞 #SunLover #DadOf2 #DogLover,"Tampa, FL",Place(_api=<tweepy.api.API object at 0x0000022...,2022-03-31 02:29:49+00:00,I am not a fan of Will Smith anymore. I pray ...,Wesley Chapel
58,Iron Boy Wonder Iron fist movie,Puerto Rico,Place(_api=<tweepy.api.API object at 0x0000022...,2022-03-31 01:38:02+00:00,@ShaunORourke5 hey can we talk about will smit...,University
83,Peter North DVDs,"Boston, MA",Place(_api=<tweepy.api.API object at 0x0000022...,2022-03-30 23:01:56+00:00,Will Smith is a Coward.,Tampa
104,Peter North DVDs,"Boston, MA",Place(_api=<tweepy.api.API object at 0x0000022...,2022-03-30 20:54:25+00:00,@MenInBlack should cancel Will Smith like Canc...,Tampa
106,kelly 💞,,Place(_api=<tweepy.api.API object at 0x0000022...,2022-03-30 20:53:31+00:00,will smith gives cal jacobs vibes 😭,Largo


In [64]:
geo_tweets

,user_name,user_loc,geotag,creation_time,text,place_name
12,I Am The One Who Knocks,South of Heaven,Place(_api=<tweepy.api.API object at 0x0000022...,2022-03-31 13:37:48+00:00,@paulduanefilm Robin Williams.\nWill Smith.\nT...,Tampa
23,444,"Florida, USA",Place(_api=<tweepy.api.API object at 0x0000022...,2022-03-31 12:15:56+00:00,Defend Will Smith although he was wrong he’s a...,Tampa
50,#UVA 🍺🏈⚾️🏀🏒🐴🌴🌞 #SunLover #DadOf2 #DogLover,"Tampa, FL",Place(_api=<tweepy.api.API object at 0x0000022...,2022-03-31 02:29:49+00:00,I am not a fan of Will Smith anymore. I pray ...,Wesley Chapel
58,Iron Boy Wonder Iron fist movie,Puerto Rico,Place(_api=<tweepy.api.API object at 0x0000022...,2022-03-31 01:38:02+00:00,@ShaunORourke5 hey can we talk about will smit...,University
83,Peter North DVDs,"Boston, MA",Place(_api=<tweepy.api.API object at 0x0000022...,2022-03-30 23:01:56+00:00,Will Smith is a Coward.,Tampa
104,Peter North DVDs,"Boston, MA",Place(_api=<tweepy.api.API object at 0x0000022...,2022-03-30 20:54:25+00:00,@MenInBlack should cancel Will Smith like Canc...,Tampa
106,kelly 💞,,Place(_api=<tweepy.api.API object at 0x0000022...,2022-03-30 20:53:31+00:00,will smith gives cal jacobs vibes 😭,Largo


Check detailed information in the geotag. The location of the geotag is in a bounding box defined by the four coordinate pairs.

In [65]:
geo_tweets.iloc[0].geotag

Place(_api=<tweepy.api.API object at 0x0000022E113CD880>, id='dc62519fda13b4ec', url='https://api.twitter.com/1.1/geo/id/dc62519fda13b4ec.json', place_type='city', name='Tampa', full_name='Tampa, FL', country_code='US', country='United States', contained_within=[], bounding_box=BoundingBox(_api=<tweepy.api.API object at 0x0000022E113CD880>, type='Polygon', coordinates=[[[-82.620093, 27.821353], [-82.2652945, 27.821353], [-82.2652945, 28.171836], [-82.620093, 28.171836]]]), attributes={})

Get coordinates of the bounding box

In [66]:
geo_tweets.iloc[0].geotag.bounding_box.coordinates[0]

[[-82.620093, 27.821353],
 [-82.2652945, 27.821353],
 [-82.2652945, 28.171836],
 [-82.620093, 28.171836]]

Create a column in the dataframe to store coordinates of the bounding boxes

In [67]:
# store bounding box coordinates to a new column
geo_tweets['bounding_box'] = geo_tweets.geotag.apply(lambda s:s.bounding_box.coordinates[0])

# print the geotag
geo_tweets.head()

,user_name,user_loc,geotag,creation_time,text,place_name,bounding_box
12,I Am The One Who Knocks,South of Heaven,Place(_api=<tweepy.api.API object at 0x0000022...,2022-03-31 13:37:48+00:00,@paulduanefilm Robin Williams.\nWill Smith.\nT...,Tampa,"[[-82.620093, 27.821353], [-82.2652945, 27.821..."
23,444,"Florida, USA",Place(_api=<tweepy.api.API object at 0x0000022...,2022-03-31 12:15:56+00:00,Defend Will Smith although he was wrong he’s a...,Tampa,"[[-82.620093, 27.821353], [-82.2652945, 27.821..."
50,#UVA 🍺🏈⚾️🏀🏒🐴🌴🌞 #SunLover #DadOf2 #DogLover,"Tampa, FL",Place(_api=<tweepy.api.API object at 0x0000022...,2022-03-31 02:29:49+00:00,I am not a fan of Will Smith anymore. I pray ...,Wesley Chapel,"[[-82.403523, 28.169963], [-82.24588, 28.16996..."
58,Iron Boy Wonder Iron fist movie,Puerto Rico,Place(_api=<tweepy.api.API object at 0x0000022...,2022-03-31 01:38:02+00:00,@ShaunORourke5 hey can we talk about will smit...,University,"[[-82.45491, 28.054805], [-82.4098119, 28.0548..."
83,Peter North DVDs,"Boston, MA",Place(_api=<tweepy.api.API object at 0x0000022...,2022-03-30 23:01:56+00:00,Will Smith is a Coward.,Tampa,"[[-82.620093, 27.821353], [-82.2652945, 27.821..."


We will use the centroids of the bounding boxes to map the tweets. The following code calculate the lat, lon of centroids and store them in two new columns.

In [68]:
geo_tweets['point']  = geo_tweets['bounding_box'].apply(lambda s: [(s[0][1]+s[2][1])/2,(s[0][0]+s[2][0])/2])

geo_tweets['lat']  = geo_tweets['bounding_box'].apply(lambda s: (s[0][1]+s[2][1])/2)

geo_tweets['lon']  = geo_tweets['bounding_box'].apply(lambda s: (s[0][0]+s[2][0])/2)

Print to see the dataframe again.

You'll see the centroids, latitude, and longitude are added as columns in the dataframe.

Note: the point column is an redundancy of the lat and lon columns. We create all these columns just for demonstration of mapping in the next step.

In [69]:
geo_tweets.head()

,user_name,user_loc,geotag,creation_time,text,place_name,bounding_box,point,lat,lon
12,I Am The One Who Knocks,South of Heaven,Place(_api=<tweepy.api.API object at 0x0000022...,2022-03-31 13:37:48+00:00,@paulduanefilm Robin Williams.\nWill Smith.\nT...,Tampa,"[[-82.620093, 27.821353], [-82.2652945, 27.821...","[27.9965945, -82.44269374999999]",27.996595,-82.442694
23,444,"Florida, USA",Place(_api=<tweepy.api.API object at 0x0000022...,2022-03-31 12:15:56+00:00,Defend Will Smith although he was wrong he’s a...,Tampa,"[[-82.620093, 27.821353], [-82.2652945, 27.821...","[27.9965945, -82.44269374999999]",27.996595,-82.442694
50,#UVA 🍺🏈⚾️🏀🏒🐴🌴🌞 #SunLover #DadOf2 #DogLover,"Tampa, FL",Place(_api=<tweepy.api.API object at 0x0000022...,2022-03-31 02:29:49+00:00,I am not a fan of Will Smith anymore. I pray ...,Wesley Chapel,"[[-82.403523, 28.169963], [-82.24588, 28.16996...","[28.2245025, -82.3247015]",28.224502,-82.324702
58,Iron Boy Wonder Iron fist movie,Puerto Rico,Place(_api=<tweepy.api.API object at 0x0000022...,2022-03-31 01:38:02+00:00,@ShaunORourke5 hey can we talk about will smit...,University,"[[-82.45491, 28.054805], [-82.4098119, 28.0548...","[28.0768615, -82.43236095]",28.076861,-82.432361
83,Peter North DVDs,"Boston, MA",Place(_api=<tweepy.api.API object at 0x0000022...,2022-03-30 23:01:56+00:00,Will Smith is a Coward.,Tampa,"[[-82.620093, 27.821353], [-82.2652945, 27.821...","[27.9965945, -82.44269374999999]",27.996595,-82.442694


---

## 5. Mapping Geotagged Tweets

Finally, we will use folium to create an interactive map for the geotagged tweets.

In [70]:
# Create a base map
maptweet = folium.Map()

# Add the tweets into the basemap
for i, row in geo_tweets.iterrows():
    folium.Marker(row.point,popup = row.text).add_to(maptweet)
    
# Zoom to the bounding box including the tweets
maptweet.fit_bounds([[min(geo_tweets.lat),min(geo_tweets.lon)],[max(geo_tweets.lat),max(geo_tweets.lon)]])

# Show the map
display(maptweet)

## 6. Streaming with Tweepy

In [75]:
#override tweepy.StreamListener to add logic to on_status
class MyStreamListener(tweepy.Stream):
    def on_status(self, status):
        print(status.text)

In [77]:
myStreamListener = MyStreamListener(API_key, API_key_secret, access_token, access_token_secret)
myStream = tweepy.Stream(auth = api.auth, listener=myStreamListener())

TypeError: 'MyStreamListener' object is not callable